In [1]:
%pip install pandas rapidfuzz

import pandas as pd
from rapidfuzz import fuzz, process
import sys
# Cell 3: Define helper functions
def get_first_two_words(text):
    """Extract first 2 words from Arabic/English text (handles short titles)"""
    if pd.isna(text) or not isinstance(text, str):
        return ""
    words = text.strip().split()
    return " ".join(words[:2]) if len(words) >= 2 else text.strip()

def merge_csvs(source_path, target_path, output_path, threshold=80):
    # Read CSVs with UTF-8-SIG encoding (handles Arabic + BOM)
    source_df = pd.read_csv(source_path, encoding='utf-8-sig')
    target_df = pd.read_csv(target_path, encoding='utf-8-sig')
    
    # Ensure required columns exist in source
    required_source_cols = ['Title', 'Singer', 'Poem', 'Date', 'YouTube', 'DetailURL']
    for col in required_source_cols:
        if col not in source_df.columns:
            raise ValueError(f"Source CSV missing required column: '{col}'")
    
    # Add new columns to target if missing (initialize as empty)
    new_cols = ['Poem_line_raw', 'Singer', 'Date', 'YouTube', 'DetailURL']
    for col in new_cols:
        if col not in target_df.columns:
            target_df[col] = pd.NA
    
    # Track matched target indices to prevent duplicates
    matched_target_indices = set()
    unmatched_source_rows = []
    
    # Create a mapping from index to title for matching
    available_targets = {
        i: str(target_df.iloc[i]['Title_raw']) 
        for i in range(len(target_df)) 
        if i not in matched_target_indices
    }
    
    # Process each source row
    for idx, row in source_df.iterrows():
        source_title = str(row['Title']).strip()
        prefix = get_first_two_words(source_title)
        
        # Skip empty prefixes
        if not prefix:
            unmatched_source_rows.append(row)
            continue
        
        # Refresh available targets (excluding already matched ones)
        available_targets = {
            i: str(target_df.iloc[i]['Title_raw']) 
            for i in range(len(target_df)) 
            if i not in matched_target_indices
        }
        
        if not available_targets:
            unmatched_source_rows.append(row)
            continue
        
        # Fuzzy match using Levenshtein ratio (prefix vs full target title)
        match_result = process.extractOne(
            prefix,
            available_targets,
            scorer=fuzz.ratio,
            score_cutoff=threshold
        )
        
        if match_result:
            matched_title, score, target_idx = match_result
            
            # EXACT MATCH: Only update Date/YouTube/DetailURL/Singer (don't overwrite poem)
            if target_df.at[target_idx, 'Title_raw'].strip() == source_title.strip():
                print(f"Exact match found for '{source_title}' - updating metadata only")
                target_df.at[target_idx, 'Singer'] = row['Singer']
                target_df.at[target_idx, 'Date'] = row['Date']
                target_df.at[target_idx, 'YouTube'] = row['YouTube']
                target_df.at[target_idx, 'DetailURL'] = row['DetailURL']
            # FUZZY MATCH: Update all fields including poem (since titles are different)
            else:
                print(f"Fuzzy match found for '{source_title}' -> '{target_df.at[target_idx, 'Title_raw']}' - updating all fields")
                target_df.at[target_idx, 'Poem_line_raw'] = row['Poem']
                target_df.at[target_idx, 'Singer'] = row['Singer']
                target_df.at[target_idx, 'Date'] = row['Date']
                target_df.at[target_idx, 'YouTube'] = row['YouTube']
                target_df.at[target_idx, 'DetailURL'] = row['DetailURL']
            
            matched_target_indices.add(target_idx)
        else:
            unmatched_source_rows.append(row)
    
    # Append unmatched source rows as NEW entries
    if unmatched_source_rows:
        new_rows = []
        for row in unmatched_source_rows:
            new_row = {col: pd.NA for col in target_df.columns}
            new_row['Title_raw'] = row['Title']  # Map Title → Title_raw for new rows
            new_row['Singer'] = row['Singer']
            new_row['Poem_line_raw'] = row['Poem']
            new_row['Date'] = row['Date']
            new_row['YouTube'] = row['YouTube']
            new_row['DetailURL'] = row['DetailURL']
            new_rows.append(new_row)
        
        # Append to target DataFrame
        target_df = pd.concat([target_df, pd.DataFrame(new_rows)], ignore_index=True)
    
    # Save result
    target_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✓ Merged {len(matched_target_indices)} rows")
    print(f"✓ Added {len(unmatched_source_rows)} new rows from source")
    print(f"✓ Output saved to: {output_path}")


# Cell 4: Set your file paths here
input_csv = ".\CSV\Diwan-Hamdan-WIP - wneen_complete.csv"      # Replace with your source CSV filename
target_csv = ".\CSV\Diwan-Hamdan-WIP - Full_poems.csv"     # Replace with your target CSV filename
output_csv = ".\CSV\merged_output.csv"  # Replace with desired output filename

# Run the merge function
merge_csvs(input_csv, target_csv, output_csv)

Note: you may need to restart the kernel to use updated packages.
Exact match found for 'دمع العظيم' - updating metadata only
Fuzzy match found for 'جسر الصداقة' -> 'جسـر الصـداقة' - updating all fields
Exact match found for 'فخر الاجيال' - updating metadata only
Fuzzy match found for 'برج عاجي ( في حضورك )' -> 'بـرج عـاجـي' - updating all fields
Fuzzy match found for 'الصاحب المعني' -> 'الصاحب الـمعني' - updating all fields
Fuzzy match found for 'ترنيمة' -> 'ترنيمــة' - updating all fields
Fuzzy match found for 'طيب القلب' -> 'طيّـب القلـب' - updating all fields
Fuzzy match found for 'انا انا' -> 'انت وانا' - updating all fields
Fuzzy match found for 'فنون الغدر' -> 'فنـون الغـدر' - updating all fields
Fuzzy match found for 'دموع المحبين' -> 'دمـوع الـمـحـبّـين' - updating all fields
Fuzzy match found for 'خلاص يعني' -> 'خـلاص يعـني' - updating all fields
Fuzzy match found for 'انا والبواخر' -> 'انا والبواخر ..' - updating all fields
Exact match found for 'رابع يوم' - updating metadat

In [20]:
# Cell 1: Install required packages
%pip install pandas rapidfuzz

# Cell 2: Import necessary libraries
import pandas as pd
from rapidfuzz import fuzz, process

# Cell 3: Define helper functions
def get_first_two_words(text):
    """Extract first 2 words from Arabic/English text (handles short titles)"""
    if pd.isna(text) or not isinstance(text, str):
        return ""
    words = text.strip().split()
    return " ".join(words[:2]) if len(words) >= 2 else text.strip()

# Cell 4: Define the merge function
def merge_csvs(source_path, target_path, output_path, threshold=80):
    """
    Merges data from source CSV into target CSV using fuzzy matching on titles.
    Updates existing rows and appends new ones. Fills missing Singer/YouTube using DetailURL.
    """
    # Read CSVs with UTF-8-SIG encoding (handles Arabic + BOM)
    source_df = pd.read_csv(source_path, encoding='utf-8-sig')
    target_df = pd.read_csv(target_path, encoding='utf-8-sig')
    
    # Ensure required columns exist in source
    required_source_cols = ['Title', 'Singer', 'Poem', 'Date', 'YouTube', 'DetailURL']
    for col in required_source_cols:
        if col not in source_df.columns:
            raise ValueError(f"Source CSV missing required column: '{col}'")
    
    # Add new columns to target if missing (initialize as empty)
    new_cols = ['Poem_line_raw', 'Singer', 'Date', 'YouTube', 'DetailURL']
    for col in new_cols:
        if col not in target_df.columns:
            target_df[col] = pd.NA
    
    # Track matched target indices to prevent duplicates
    matched_target_indices = set()
    unmatched_source_rows = []
    
    # Create a mapping from index to title for matching
    available_targets = {
        i: str(target_df.iloc[i]['Title_raw']) 
        for i in range(len(target_df)) 
        if i not in matched_target_indices
    }
    
    # Process each source row
    for idx, row in source_df.iterrows():
        source_title = str(row['Title']).strip()
        prefix = get_first_two_words(source_title)
        
        # Skip empty prefixes
        if not prefix:
            unmatched_source_rows.append(row)
            continue
        
        # Refresh available targets (excluding already matched ones)
        available_targets = {
            i: str(target_df.iloc[i]['Title_raw']) 
            for i in range(len(target_df)) 
            if i not in matched_target_indices
        }
        
        if not available_targets:
            unmatched_source_rows.append(row)
            continue
        
        # Fuzzy match using Levenshtein ratio (prefix vs full target title)
        match_result = process.extractOne(
            prefix,
            available_targets,
            scorer=fuzz.ratio,
            score_cutoff=threshold
        )
        
        if match_result:
            matched_title, score, target_idx = match_result
            
            # EXACT MATCH: Only update Date/YouTube/DetailURL/Singer (don't overwrite poem)
            if target_df.at[target_idx, 'Title_raw'].strip() == source_title.strip():
                print(f"Exact match found for '{source_title}' - updating metadata only")
                target_df.at[target_idx, 'Singer'] = row['Singer']
                target_df.at[target_idx, 'Date'] = row['Date']
                target_df.at[target_idx, 'YouTube'] = row['YouTube']
                target_df.at[target_idx, 'DetailURL'] = row['DetailURL']
            # FUZZY MATCH: Update all fields including poem (since titles are different)
            else:
                print(f"Fuzzy match found for '{source_title}' -> '{target_df.at[target_idx, 'Title_raw']}' - updating all fields")
                target_df.at[target_idx, 'Poem_line_raw'] = row['Poem']
                target_df.at[target_idx, 'Singer'] = row['Singer']
                target_df.at[target_idx, 'Date'] = row['Date']
                target_df.at[target_idx, 'YouTube'] = row['YouTube']
                target_df.at[target_idx, 'DetailURL'] = row['DetailURL']
            
            matched_target_indices.add(target_idx)
        else:
            unmatched_source_rows.append(row)
    
    # Append unmatched source rows as NEW entries
    if unmatched_source_rows:
        new_rows = []
        for row in unmatched_source_rows:
            new_row = {col: pd.NA for col in target_df.columns}
            new_row['Title_raw'] = row['Title']  # Map Title → Title_raw for new rows
            new_row['Poem_line_raw'] = row['Poem']
            new_row['Singer'] = row['Singer']
            new_row['Date'] = row['Date']
            new_row['YouTube'] = row['YouTube']
            new_row['DetailURL'] = row['DetailURL']
            print(f"New row created: \"{row['Title']}\" | Singer: {row.get('Singer', 'MISSING')}")
            new_rows.append(new_row)
        
        # Append to target DataFrame
        target_df = pd.concat([target_df, pd.DataFrame(new_rows)], ignore_index=True)
    
    # Fill missing Singer & YouTube using DetailURL lookup
    singer_lookup = dict(zip(source_df['DetailURL'], source_df['Singer']))
    youtube_lookup = dict(zip(source_df['DetailURL'], source_df['YouTube']))
    
    updates = 0
    for idx, row in target_df.iterrows():
        url = row['DetailURL']
        if pd.notna(url):
            if pd.isna(row.get('Singer')) and url in singer_lookup:
                target_df.at[idx, 'Singer'] = singer_lookup[url]
                updates += 1
            if pd.isna(row.get('YouTube')) and url in youtube_lookup:
                target_df.at[idx, 'YouTube'] = youtube_lookup[url]
                updates += 1
    
    print(f"Filled {updates} missing Singer/YouTube values")
    
    print("\nFinal check - rows with Singer still empty:")
    empty_singer = target_df[target_df['Singer'].isna()]
    if not empty_singer.empty:
        for _, r in empty_singer.head(10).iterrows():  # limit to first 10
            print(f"  - \"{r['Title_raw']}\" | DetailURL: {r.get('DetailURL', 'none')}")
        print(f"Total rows missing Singer: {len(empty_singer)}")
    else:
        print("All rows now have Singer filled.")
    
    # Save result
    target_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✓ Merged {len(matched_target_indices)} rows")
    print(f"✓ Added {len(unmatched_source_rows)} new rows from source")
    print(f"✓ Output saved to: {output_path}")

# Cell 5: Set your file paths here
input_csv = ".\\CSV\\Diwan-Hamdan-WIP - wneen_complete.csv"      # Replace with your source CSV filename
target_csv = ".\\CSV\\Diwan-Hamdan-WIP - Full_poems.csv"     # Replace with your target CSV filename
output_csv = ".\\CSV\\merged_output.csv"  # Replace with desired output filename

# Run the merge function
merge_csvs(input_csv, target_csv, output_csv)

Note: you may need to restart the kernel to use updated packages.
Exact match found for 'دمع العظيم' - updating metadata only
Fuzzy match found for 'جسر الصداقة' -> 'جسـر الصـداقة' - updating all fields
Exact match found for 'فخر الاجيال' - updating metadata only
Fuzzy match found for 'برج عاجي ( في حضورك )' -> 'بـرج عـاجـي' - updating all fields
Fuzzy match found for 'الصاحب المعني' -> 'الصاحب الـمعني' - updating all fields
Fuzzy match found for 'ترنيمة' -> 'ترنيمــة' - updating all fields
Fuzzy match found for 'طيب القلب' -> 'طيّـب القلـب' - updating all fields
Fuzzy match found for 'انا انا' -> 'انت وانا' - updating all fields
Fuzzy match found for 'فنون الغدر' -> 'فنـون الغـدر' - updating all fields
Fuzzy match found for 'دموع المحبين' -> 'دمـوع الـمـحـبّـين' - updating all fields
Fuzzy match found for 'خلاص يعني' -> 'خـلاص يعـني' - updating all fields
Fuzzy match found for 'انا والبواخر' -> 'انا والبواخر ..' - updating all fields
Exact match found for 'رابع يوم' - updating metadat

In [ ]:
# pip install yt-dlp

import yt_dlp
import os
import pandas as pd

df = pd.read_csv("./CSV/Diwan-Hamdan-WIP - Full_poems.csv")



for _, row in df.iterrows():
    url = row['YouTube']
    if pd.isna(url) or not isinstance(url, str) or not url.strip():
        print(f"Skipping invalid URL for poem_id {row['poem_id']}")
        continue

    pid = str(row['poem_id'])
    title = str(row['Titlecleaned']).strip()
    singer = str(row['Singer']).strip()
    date = str(row['Date']).strip()

    if len(date) == 5 and date.endswith('0'):
        date = date[:-1]

    base = f"{pid}_{title}_{singer}_{date}"
    base = "".join(c for c in base if c.isalnum() or c in " -_").strip()

    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/bestvideo+bestaudio/best',
        'outtmpl': f"{base}.%(ext)s",
        'merge_output_format': 'mp4',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'keepvideo': True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            ydl.download([url])
        except Exception as e:
            print(f"Failed {base}: {e}")

In [12]:
import yt_dlp
import pandas as pd

df = pd.read_csv("./CSV/Diwan-Hamdan-WIP - Full_poems.csv")


for _, row in df.iterrows():
    url = row.get('YouTube')
    if pd.isna(url) or not isinstance(url, str) or not url.strip():
        continue

    title_raw = row.get('Title_raw')
    has_title_raw = not (pd.isna(title_raw) or str(title_raw).strip() == '')

    date = str(row.get('Date', '')).strip()
    if len(date) == 5 and date.endswith('0'):
        date = date[:-1]
    if not date or date in ('20090', '20100', '20110'):  # skip bad dates
        date = ''

    if has_title_raw:
        title = str(title_raw).strip()
        singer = str(row.get('Singer', '')).strip()
        base = f"{title}_{singer}"
        if date:
            base += f"_{date}"
    else:
        pid = str(row.get('poem_id', '')).strip()
        if not pid:
            continue  # skip if poem_id also missing
        title_cleaned = str(row.get('Titlecleaned', '')).strip()
        singer = str(row.get('Singer', '')).strip()
        base = f"{pid}_{title_cleaned}_{singer}"
        if date:
            base += f"_{date}"

    base = "".join(c for c in base if c.isalnum() or c in " -_").strip()

    ydl_opts = {
        'format': 'bestvideo[ext=mp4]/best[ext=mp4]/best',
        'outtmpl': f"{base}.%(ext)s",
        'merge_output_format': 'mp4',
        'nopostoverwrites': True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

[youtube] Extracting URL: https://www.youtube.com/watch?v=g43mbNGxiSk
[youtube] g43mbNGxiSk: Downloading webpage


[youtube] g43mbNGxiSk: Downloading android vr player API JSON
[info] g43mbNGxiSk: Downloading 1 format(s): 135
[download] Destination: ثــلاث مـرات_اسماء لمنور و حسين الجسمي_20090.mp4
[download] 100% of   38.91MiB in 00:00:00 at 71.73MiB/s    
[youtube] Extracting URL: https://www.youtube.com/watch?v=MpHlpNuLr2U
[youtube] MpHlpNuLr2U: Downloading webpage


[youtube] MpHlpNuLr2U: Downloading android vr player API JSON
[info] MpHlpNuLr2U: Downloading 1 format(s): 135
[download] Destination: يتيم_حسين الجسمي و ميحد حمد_20110.mp4
[download] 100% of   15.50MiB in 00:00:00 at 56.75MiB/s    
[youtube] Extracting URL: https://www.youtube.com/watch?v=ofkceALh_po
[youtube] ofkceALh_po: Downloading webpage


[youtube] ofkceALh_po: Downloading android vr player API JSON
[info] ofkceALh_po: Downloading 1 format(s): 137
[download] Destination: سامح_ابوبكر سالم و راشد الماجد_20100.mp4
[download] 100% of    7.33MiB in 00:00:00 at 45.01MiB/s  
[youtube] Extracting URL: https://www.youtube.com/watch?v=4ipBiQSkVxA
[youtube] 4ipBiQSkVxA: Downloading webpage


KeyboardInterrupt: 

In [ ]:
import yt_dlp
import pandas as pd

df = pd.read_csv("./CSV/Diwan-Hamdan-WIP - Full_poems.csv")


for idx, row in df.iterrows():
    url = row.get('YouTube')
    youtube_exists = not (pd.isna(url) or not str(url).strip())

    title_raw = row.get('Title_raw')
    title_raw_exists = not (pd.isna(title_raw) or str(title_raw).strip() == '')

    if not youtube_exists or title_raw_exists:
        continue  # only proceed if YouTube exists AND Title_raw does NOT exist

    pid         = str(row.get('poem_id', '')).strip()
    title_clean = str(row.get('Titlecleaned', '')).strip()
    singer      = str(row.get('Singer', '')).strip()
    date_raw    = row.get('Date', '')
    date        = str(date_raw).strip()

    if len(date) == 5 and date.endswith('0'):
        date = date[:-1]
    if date in ('', '20090', '20100', '20110'):
        date = ''

    base = f"{pid}_{title_clean}_{singer}"
    if date:
        base += f"_{date}"

    base = "".join(c for c in base if c.isalnum() or c in " -_").strip()
    simulated = f"{base}.mp4"

    # print(f"Row {idx} → {simulated}")
    # print(f"   YouTube     = {url}")
    # print(f"   Title_raw   = {title_raw}")
    # print(f"   poem_id     = {pid}")
    # print(f"   Titlecleaned= {title_clean}")
    # print(f"   Singer      = {singer}")
    # print(f"   Date raw    = {date_raw} → {date}")
    # print("-" * 50)
    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
        'outtmpl': f"{base}.%(ext)s",
        'merge_output_format': 'mp4',
        'nopostoverwrites': True,
    }

    print(f"Starting download row {idx}: {base}.mp4")
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            ydl.download([url])
            print(f"Success: {base}.mp4")
        except Exception as e:
            print(f"Error row {idx}: {e}")

In [1]:
import os
import re

folder = "D:\Jobs\HH\Poem\POETRY-rag\poetry-dashboard\TODO\SHBMPoetryMedia\Singers"  # ← change this

for filename in os.listdir(folder):
    if not filename.endswith(".mp4"):
        continue

    path = os.path.join(folder, filename)
    base, ext = os.path.splitext(filename)

    parts = base.rsplit("_", 1)
    if len(parts) < 2:
        continue

    prefix, last = parts
    new_last = last

    if last == "00":
        new_last = "0"
    elif last == "0":
        pass  # already correct
    elif last.endswith("0") and len(last) in (4, 5):
        new_last = last[:-1]

    prefix_parts = prefix.split("_", 1)
    if len(prefix_parts) >= 1:
        pid = prefix_parts[0]
        if pid.isdigit() and len(pid) > 2 and pid.endswith("0"):
            new_pid = pid[:-1]
            prefix_parts[0] = new_pid
            prefix = "_".join(prefix_parts)

    new_base = prefix
    if new_last:
        new_base += f"_{new_last}"

    new_filename = f"{new_base}{ext}"
    new_path = os.path.join(folder, new_filename)

    if new_filename != filename:
        if os.path.exists(new_path):
            print(f"Skip (exists): {new_filename}")
        else:
            os.rename(path, new_path)
            print(f"Renamed: {filename} → {new_filename}")

Renamed: 1010_دموع المحبين_ميحد حمد_00.mp4 → 101_دموع المحبين_ميحد حمد_0.mp4
Renamed: 1020_خير الكلام_ابوبكر سالم و حسين الجسمي_20110.mp4 → 102_خير الكلام_ابوبكر سالم و حسين الجسمي_2011.mp4
Renamed: 1030_ديره الغربه_اصاله نصري_20070.mp4 → 103_ديره الغربه_اصاله نصري_2007.mp4
Renamed: 120_سامح_ابوبكر سالم و راشد الماجد_20100.mp4 → 12_سامح_ابوبكر سالم و راشد الماجد_2010.mp4
Renamed: 1220_شياطينك_راشد الماجد_20050.mp4 → 122_شياطينك_راشد الماجد_2005.mp4
Renamed: 1230_صبري ملني_راشد الماجد_20040.mp4 → 123_صبري ملني_راشد الماجد_2004.mp4
Renamed: 1290_طيب القلب_ميحد حمد_00.mp4 → 129_طيب القلب_ميحد حمد_0.mp4
Renamed: 1410_فخر الاجيال_حسين الجسمي_20250.mp4 → 141_فخر الاجيال_حسين الجسمي_2025.mp4
Renamed: 1450_فمان الله_محمد عبده_00.mp4 → 145_فمان الله_محمد عبده_0.mp4
Renamed: 1460_فنون الغدر_ميحد حمد_00.mp4 → 146_فنون الغدر_ميحد حمد_0.mp4
Renamed: 150_جسر الصداقه_الوسمي_20160.mp4 → 15_جسر الصداقه_الوسمي_2016.mp4
Renamed: 1590_كحل الظلام_ابوبكر سالم و اسماء لمنور_00.mp4 → 159_كحل الظلام_ابوبكر سال

In [ ]:
import yt_dlp
import pandas as pd
import time

# ===== Load CSV =====
df = pd.read_csv("./CSV/Diwan-Hamdan-WIP - Full_poems.csv")

# ===== yt-dlp base options =====
BASE_YDL_OPTS = {
    # Best mp4 combo
    'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
    'merge_output_format': 'mp4',

    # Output
    'outtmpl': '%(title)s.%(ext)s',
    'nopostoverwrites': True,

    # ✅ AUTH: use your real browser login
    'cookiesfrombrowser': ('chrome',),  # change to 'firefox' or 'edge' if needed

    # 🧠 Anti-bot
    'sleep_interval': 5,
    'max_sleep_interval': 10,
    'concurrent_fragment_downloads': 1,

    # 🧼 No weird cache
    'cachedir': False,

    # 🛡️ Real browser fingerprint
    'http_headers': {
        'User-Agent':
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/121.0.0.0 Safari/537.36'
    },

    # Cleaner logs
    'quiet': True,
    'no_warnings': True,
}

# ===== Main Loop =====
for idx, row in df.iterrows():
    url = row.get('YouTube')
    title_raw = row.get('Title_raw')

    youtube_exists = not (pd.isna(url) or not str(url).strip())
    title_raw_exists = not (pd.isna(title_raw) or str(title_raw).strip())

    # Only download if YouTube exists AND Title_raw does NOT exist
    if not youtube_exists or title_raw_exists:
        continue

    pid         = str(row.get('poem_id', '')).strip()
    title_clean = str(row.get('Titlecleaned', '')).strip()
    singer      = str(row.get('Singer', '')).strip()
    date_raw    = str(row.get('Date', '')).strip()

    # Clean date
    if len(date_raw) == 5 and date_raw.endswith('0'):
        date_raw = date_raw[:-1]
    if date_raw in ('', '20090', '20100', '20110'):
        date_raw = ''

    # Build filename
    base = f"{pid}_{title_clean}_{singer}"
    if date_raw:
        base += f"_{date_raw}"

    base = "".join(c for c in base if c.isalnum() or c in " -_").strip()

    ydl_opts = BASE_YDL_OPTS.copy()
    ydl_opts['outtmpl'] = f"{base}.%(ext)s"

    print(f"▶️  Row {idx}: {base}.mp4")

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])

        print(f"✅ Success: {base}.mp4")

    except Exception as e:
        print(f"❌ Error row {idx}: {e}")

    # ⏳ Mandatory cooldown between videos
    time.sleep(8)

print("🎉 All downloads completed.")


▶️  Row 1: 2_ثلاث مرات_اسماء لمنور و حسين الجسمي_20090.mp4


ERROR: Could not copy Chrome cookie database. See  https://github.com/yt-dlp/yt-dlp/issues/7271  for more info
ERROR: ERROR: Could not copy Chrome cookie database. See  https://github.com/yt-dlp/yt-dlp/issues/7271  for more info


❌ Error row 1: ERROR: ERROR: Could not copy Chrome cookie database. See  https://github.com/yt-dlp/yt-dlp/issues/7271  for more info


ERROR: Could not copy Chrome cookie database. See  https://github.com/yt-dlp/yt-dlp/issues/7271  for more info
ERROR: ERROR: Could not copy Chrome cookie database. See  https://github.com/yt-dlp/yt-dlp/issues/7271  for more info


▶️  Row 3: 4_يتيم_حسين الجسمي و ميحد حمد_20110.mp4
❌ Error row 3: ERROR: ERROR: Could not copy Chrome cookie database. See  https://github.com/yt-dlp/yt-dlp/issues/7271  for more info
